# Modèle 2 : durée du séjour de retour à l'hôpital

**Objectif :** pour un patient qui sort, prédire **combien de jours il restera hospitalisé s'il revient**, avec une fourchette (de 10 % à 90 %), pour calculer les lits à réserver.

**Ce que ce modèle est (et n'est pas) :** il est entraîné uniquement sur des patients **qui sont revenus**. Sa réponse est donc toujours conditionnelle : « *s'il revient*, il restera environ X jours ». La probabilité de revenir vient du modèle 1. Le calcul final est :

> charge attendue = probabilité de retour (modèle 1) × durée moyenne prévue (modèle 2)

**Comment l'utiliser dans Google Colab :**
1. Ouvrez [colab.research.google.com](https://colab.research.google.com), *Fichier → Importer un notebook*, choisissez ce fichier.
2. Importez `diabetic_data.csv` (panneau Fichiers à gauche, icône d'import) ou laissez l'étape 1 vous le demander.
3. Menu *Exécution → Tout exécuter* (ou cellule par cellule avec Maj+Entrée).

Le jeu de données est petit (quelques milliers de couples), donc il n'y a pas besoin d'échantillon pour un test rapide : tout s'exécute en **5 à 15 minutes** (estimation, CatBoost est le plus lent).

| Étape | Contenu |
|---|---|
| 1 | Installation, réglages, import du fichier |
| 2 | Construction des couples « séjour → séjour de retour » |
| 3 | Création des variables |
| 4 | Séparation entraînement / test **par patient** |
| 5 | Étude comparative de plusieurs modèles |
| 6 | Faut-il ajouter les retours de plus de 30 jours ? |
| 7 | Réglage rapide |
| 8 | Modèles finaux (médiane, fourchette, moyenne) et test final |
| 9 | Où le modèle se trompe |
| 10 | Importance des variables |
| 11 | Sauvegarde, utilisation et charge attendue |

## Étape 1. Installation, réglages, import du fichier

In [ ]:
import os, sys, time, subprocess, warnings, joblib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# ---- Réglages ----
SAMPLE = None          # None = toutes les données ; ex. 15000 = test de code sur 15 000 patients
SEED = 42
INSTALL_EXTRAS = True
DATA_PATH = "diabetic_data.csv"
N_REP = 3              # nombre de répétitions de la validation croisée (données petites = résultats bruités)

def ensure(pkg, import_name=None):
    try:
        __import__(import_name or pkg); return True
    except ImportError:
        if not INSTALL_EXTRAS: return False
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
        try:
            __import__(import_name or pkg); return True
        except ImportError:
            return False

HAS_LGBM = ensure("lightgbm")
HAS_XGB  = ensure("xgboost")
HAS_CAT  = ensure("catboost")
print("LightGBM:", HAS_LGBM, "| XGBoost:", HAS_XGB, "| CatBoost:", HAS_CAT)
assert HAS_LGBM, "LightGBM est nécessaire pour les étapes 7 à 11 (pip install lightgbm)."

try:
    from google.colab import files
    if not os.path.exists(DATA_PATH):
        print("Importez diabetic_data.csv :")
        files.upload()
except ImportError:
    pass

## Étape 2. Construction des couples « séjour → séjour de retour »

Le fichier n'a pas de colonne « durée du retour ». Mais il contient plusieurs séjours par patient. On trie les séjours de chaque patient dans l'ordre (`patient_nbr`, puis `encounter_id`, qui est croissant pour tous les patients) et, pour chaque séjour, on lit la durée du **séjour suivant**.

| Séjour | Patient | Durée | readmitted | Cible (durée du séjour suivant) |
|---|---|---|---|---|
| 1er séjour | 1042 | 3 jours | `<30` | **6 jours** (durée du 2e séjour) |
| 2e séjour | 1042 | 6 jours | `NO` | non utilisé |

*(exemple illustratif)*

On garde les séjours marqués `<30` ou `>30` qui ont un séjour suivant. On retire :
- les séjours marqués `NO` (même s'ils ont un séjour suivant : 490 cas incohérents) ;
- les patients décédés ou en soins palliatifs à la sortie (ils ne peuvent pas être réadmis).

In [ ]:
raw = pd.read_csv(DATA_PATH, na_values="?", low_memory=False)
if SAMPLE:
    keep = raw["patient_nbr"].drop_duplicates().sample(SAMPLE, random_state=SEED)
    raw = raw[raw["patient_nbr"].isin(keep)].copy()

raw = raw.sort_values(["patient_nbr", "encounter_id"]).reset_index(drop=True)
raw["next_los"] = raw.groupby("patient_nbr")["time_in_hospital"].shift(-1)

EXCLUDE_DISCH = [11, 13, 14, 19, 20, 21]
pairs = raw[(raw["readmitted"] != "NO") & raw["next_los"].notna()
            & ~raw["discharge_disposition_id"].isin(EXCLUDE_DISCH)].copy().reset_index(drop=True)
pairs["is_lt30"] = (pairs["readmitted"] == "<30").astype(int)

print("Séjours dans le fichier :", len(raw))
print("Couples utilisables      :", len(pairs))
print("   dont retour < 30 jours :", int(pairs["is_lt30"].sum()), "(cible principale)")
print("   dont retour > 30 jours :", int((1 - pairs["is_lt30"]).sum()))

lt = pairs[pairs["is_lt30"] == 1]
print("\nDurée du séjour de retour (retours < 30 jours) :")
print(lt["next_los"].describe().round(2).to_string())
print("Médiane :", lt["next_los"].median(), "jours | moyenne :", round(lt["next_los"].mean(), 2), "jours")
print("Corrélation entre la durée du 1er séjour et celle du retour :",
      round(lt[["time_in_hospital", "next_los"]].corr().iloc[0, 1], 3))

**Comment lire :** la durée du séjour de retour varie beaucoup (de 1 à 14 jours). Si la corrélation avec la durée du premier séjour est faible, on ne peut pas s'attendre à une grande précision : le gain sur une règle simple (« toujours la médiane ») sera modeste.

## Étape 3. Création des variables

Les variables sont celles du **séjour qui se termine** (le séjour précédent), les mêmes que pour le modèle 1. Aucune information du séjour de retour n'est utilisée, sinon le modèle « tricherait ».

On ajoute une variable `retour_lt30` (1 si le retour a eu lieu en moins de 30 jours). Elle ne sert que si on ajoute les retours de plus de 30 jours à l'entraînement (étape 6). À l'utilisation, on la met toujours à 1.

In [ ]:
MED_COLS = ["metformin","repaglinide","nateglinide","chlorpropamide","glimepiride","acetohexamide",
            "glipizide","glyburide","tolbutamide","pioglitazone","rosiglitazone","acarbose","miglitol",
            "troglitazone","tolazamide","examide","citoglipton","insulin","glyburide-metformin",
            "glipizide-metformin","glimepiride-pioglitazone","metformin-rosiglitazone","metformin-pioglitazone"]

def icd9_group(code):
    if pd.isna(code): return "inconnu"
    c = str(code)
    if c.startswith(("E", "V")): return "autre"
    n = float(c)
    if int(n) == 250: return "diabete"
    if 390 <= n <= 459 or int(n) == 785: return "circulatoire"
    if 460 <= n <= 519 or int(n) == 786: return "respiratoire"
    if 520 <= n <= 579 or int(n) == 787: return "digestif"
    if 800 <= n <= 999: return "blessure"
    if 710 <= n <= 739: return "musculo-squelettique"
    if 580 <= n <= 629 or int(n) == 788: return "genito-urinaire"
    if 140 <= n <= 239: return "cancer"
    return "autre"

CAT = ["race","gender","payer_code","specialite","diag_1_grp","diag_2_grp","diag_3_grp","A1Cresult",
       "max_glu_serum","insulin","metformin","change","diabetesMed","admission_type_id",
       "admission_source_id","discharge_disposition_id"]
NUM = ["age_num","number_outpatient","number_emergency","number_inpatient","visites_anterieures",
       "number_diagnoses","time_in_hospital","num_lab_procedures","num_procedures","num_medications",
       "n_meds_diabete","n_meds_modifies","retour_lt30"]

TOP_SPEC = pairs["medical_specialty"].value_counts().head(15).index.tolist()

def build_features(d, retour_lt30=None):
    """Variables du séjour qui se termine. retour_lt30 : None = lire is_lt30 ; sinon valeur imposée (1 à l'utilisation)."""
    d = d.copy()
    for c in ["diag_1", "diag_2", "diag_3"]:
        d[c + "_grp"] = d[c].map(icd9_group)
    d["age_num"] = d["age"].str.extract(r"\[(\d+)-")[0].astype(float) + 5
    d["n_meds_diabete"]  = (d[MED_COLS] != "No").sum(axis=1)
    d["n_meds_modifies"] = d[MED_COLS].isin(["Up", "Down"]).sum(axis=1)
    d["visites_anterieures"] = d["number_outpatient"] + d["number_emergency"] + d["number_inpatient"]
    d["specialite"] = d["medical_specialty"].where(d["medical_specialty"].isin(TOP_SPEC), "autre").fillna("inconnue")
    for c, v in [("payer_code","inconnu"),("race","inconnu"),("A1Cresult","non_mesure"),("max_glu_serum","non_mesure")]:
        d[c] = d[c].fillna(v)
    d["retour_lt30"] = d["is_lt30"] if retour_lt30 is None else retour_lt30
    X = d[CAT + NUM].copy()
    for c in ["admission_type_id", "admission_source_id", "discharge_disposition_id"]:
        X[c] = X[c].astype(str)
    return X

X_raw = build_features(pairs)
CATEGORIES = {c: sorted(X_raw[c].astype(str).unique()) for c in CAT}

def apply_categories(X):
    X = X.copy()
    for c in CAT:
        X[c] = pd.Categorical(X[c].astype(str), categories=CATEGORIES[c])
    return X

X = apply_categories(X_raw)
y = pairs["next_los"].values.astype(float)
groups = pairs["patient_nbr"].values
lt30 = pairs["is_lt30"].values
print(X.shape, "| valeurs manquantes :", int(X.isna().sum().sum()))
X.head(3)

## Étape 4. Séparation entraînement / test, par patient

Un patient a souvent plusieurs séjours (donc plusieurs couples). Il ne doit jamais être à la fois dans l'entraînement et dans le test.

- **20 % des patients** sont mis de côté comme **test final** (étape 8).
- Les 80 % restants (`dev`) servent à comparer et régler les modèles.
- Toutes les mesures portent sur les **retours en moins de 30 jours** (`retour_lt30 = 1`), qui sont ceux qui comptent pour la planification.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold

dev_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED).split(X, y, groups))
X_dev,  y_dev,  g_dev,  lt_dev  = X.iloc[dev_idx].reset_index(drop=True),  y[dev_idx],  groups[dev_idx],  lt30[dev_idx]
X_test, y_test, g_test, lt_test = X.iloc[test_idx].reset_index(drop=True), y[test_idx], groups[test_idx], lt30[test_idx]

assert len(set(g_dev) & set(g_test)) == 0, "Fuite : un patient est dans dev ET test"
print("dev :", len(X_dev), "couples dont", int(lt_dev.sum()), "retours < 30 j |", len(set(g_dev)), "patients")
print("test:", len(X_test), "couples dont", int(lt_test.sum()), "retours < 30 j |", len(set(g_test)), "patients")

## Étape 5. Étude comparative

Tous les modèles reçoivent les mêmes données et les mêmes découpages (validation croisée à 5 plis, groupée par patient, **répétée 3 fois** : avec peu de données, un seul découpage donne un résultat trop bruité). Ils apprennent à prédire la **médiane** de la durée (perte « erreur absolue »).

- **MAE** : erreur moyenne en jours (plus petit = mieux). C'est la mesure principale.
- **Gain** : de combien la MAE baisse par rapport à la référence « toujours la médiane ».
- **± 1 jour / ± 2 jours** : part des prédictions (arrondies au jour) à moins de 1 ou 2 jours de la réalité.

Ici on entraîne et on évalue uniquement sur les retours en moins de 30 jours.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor

def bucket(v):  # 0 = court (1-2 j), 1 = normal (3-6 j), 2 = long (> 6 j)
    return np.where(v <= 2, 0, np.where(v <= 6, 1, 2))

def prep(name, Xd):
    if name.startswith("CatBoost"):
        Xd = Xd.copy()
        for c in CAT: Xd[c] = Xd[c].astype(str)
    return Xd

def onehot_pipe(model):
    return Pipeline([("prep", ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30), CAT),
        ("num", StandardScaler(), NUM)])), ("reg", model)])

factories = {}
factories["Référence (médiane)"] = lambda: DummyRegressor(strategy="median")
factories["Ridge"] = lambda: onehot_pipe(Ridge(alpha=30.0))
factories["RandomForest"] = lambda: onehot_pipe(RandomForestRegressor(
    n_estimators=300, min_samples_leaf=30, max_features=0.5, n_jobs=-1, random_state=SEED))
factories["HistGB"] = lambda: HistGradientBoostingRegressor(
    loss="absolute_error", max_depth=4, learning_rate=0.05, max_iter=300, l2_regularization=5.0,
    min_samples_leaf=40, categorical_features="from_dtype", random_state=SEED)

from lightgbm import LGBMRegressor
factories["LightGBM"] = lambda: LGBMRegressor(
    objective="l1", n_estimators=300, learning_rate=0.05, num_leaves=7, min_child_samples=40, reg_lambda=5.0,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=SEED, n_jobs=-1, verbose=-1)

if HAS_XGB:
    from xgboost import XGBRegressor
    factories["XGBoost"] = lambda: XGBRegressor(
        objective="reg:absoluteerror", n_estimators=300, learning_rate=0.05, max_depth=3, min_child_weight=5,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=5.0, tree_method="hist", enable_categorical=True,
        random_state=SEED, n_jobs=-1)

if HAS_CAT:
    from catboost import CatBoostRegressor
    factories["CatBoost"] = lambda: CatBoostRegressor(
        loss_function="MAE", iterations=400, learning_rate=0.06, depth=4, l2_leaf_reg=5.0, cat_features=CAT,
        random_seed=SEED, verbose=0, thread_count=-1)

def cv_los(name, factory, scope="lt30", n_rep=N_REP, n_splits=5):
    """Validation croisée groupée par patient. scope : 'lt30' = entraîner sur les retours < 30 j ;
    'all' = entraîner sur tous les retours. L'évaluation porte toujours sur les retours < 30 j."""
    maes, w1, w2, t0 = [], [], [], time.time()
    for rep in range(n_rep):
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED + rep)
        for tr, va in cv.split(X_dev, bucket(y_dev), groups=g_dev):
            tr = tr if scope == "all" else tr[lt_dev[tr] == 1]
            va = va[lt_dev[va] == 1]
            m = factory()
            m.fit(prep(name, X_dev.iloc[tr]), y_dev[tr])
            p = np.clip(m.predict(prep(name, X_dev.iloc[va])), 1, 14)
            err = np.abs(p - y_dev[va]); err_j = np.abs(np.round(p) - y_dev[va])   # ± jours : durées arrondies
            maes.append(err.mean()); w1.append((err_j <= 1).mean()); w2.append((err_j <= 2).mean())
    return dict(MAE_moy=np.mean(maes), MAE_ecart=np.std(maes), Dans_1j=np.mean(w1), Dans_2j=np.mean(w2),
                Temps_s=time.time() - t0)

REF = "Référence (médiane)"
results = {}
for name in list(factories):
    results[name] = cv_los(name, factories[name], scope="lt30")
    print(f"{name:20s} MAE {results[name]['MAE_moy']:.3f} ± {results[name]['MAE_ecart']:.3f} jours | "
          f"{results[name]['Temps_s']:.0f}s")

comparison = pd.DataFrame(results).T.sort_values("MAE_moy")
comparison["Gain_vs_ref_%"] = ((results[REF]["MAE_moy"] - comparison["MAE_moy"]) / results[REF]["MAE_moy"] * 100)
comparison.round(3)

**Comment choisir :** prenez la plus petite MAE. Si deux modèles sont à moins d'un écart-type l'un de l'autre, prenez le **plus rapide et le plus simple**. Si le meilleur modèle ne bat pas clairement la référence « toujours la médiane », il faut le dire : la durée du séjour de retour est alors surtout imprévisible avec ces données, et la fourchette est plus utile que le chiffre central.

## Étape 6. Faut-il ajouter les retours de plus de 30 jours à l'entraînement ?

Les retours en moins de 30 jours sont peu nombreux. Les retours de plus de 30 jours (`>30`) sont plus nombreux : s'ils ressemblent aux premiers, ils peuvent aider le modèle à apprendre. On compare deux façons d'entraîner **le même modèle**, toujours évaluées sur les retours en moins de 30 jours :

- `lt30` : on entraîne uniquement sur les retours en moins de 30 jours ;
- `all` : on entraîne sur tous les retours, en indiquant à chaque fois si c'est un retour rapide (`retour_lt30`).

**Règle :** on garde `lt30` (plus simple et plus fidèle à l'usage) sauf si `all` fait baisser la MAE d'au moins 0,02 jour.

In [ ]:
cand = [n for n in comparison.index if n != REF]
BEST_NAME = cand[0]                      # ou forcez-le, par exemple : BEST_NAME = "HistGB"
print("Modèle de l'étape 5 utilisé pour ce test :", BEST_NAME)

scope_res = {s: cv_los(BEST_NAME, factories[BEST_NAME], scope=s) for s in ["lt30", "all"]}
scope_tab = pd.DataFrame(scope_res).T.round(3)
display(scope_tab)

SCOPE = "all" if scope_res["all"]["MAE_moy"] < scope_res["lt30"]["MAE_moy"] - 0.02 else "lt30"
print("Données d'entraînement retenues :", SCOPE)

## Étape 7. Réglage rapide

On teste 4 réglages de LightGBM (arbres petits, forte régularisation : le signal est faible et le jeu de données petit, donc le risque de surapprentissage est élevé) avec les données d'entraînement retenues à l'étape 6.

In [ ]:
grid = {
    "A": dict(num_leaves=4,  min_child_samples=60, learning_rate=0.05, n_estimators=200, reg_lambda=10.0),
    "B": dict(num_leaves=7,  min_child_samples=40, learning_rate=0.05, n_estimators=300, reg_lambda=5.0),
    "C": dict(num_leaves=7,  min_child_samples=80, learning_rate=0.03, n_estimators=400, reg_lambda=10.0),
    "D": dict(num_leaves=15, min_child_samples=60, learning_rate=0.03, n_estimators=300, reg_lambda=10.0),
}
def make_lgbm(params, objective="l1", alpha=None):
    p = dict(params)
    if alpha is not None: p["alpha"] = alpha
    return LGBMRegressor(objective=objective, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                         random_state=SEED, n_jobs=-1, verbose=-1, **p)

tune = {}
for g, params in grid.items():
    tune[f"LGBM_{g}"] = cv_los("LightGBM", (lambda p=params: make_lgbm(p)), scope=SCOPE)
    print(f"LGBM_{g}: MAE {tune[f'LGBM_{g}']['MAE_moy']:.3f} ± {tune[f'LGBM_{g}']['MAE_ecart']:.3f}")
tune_tab = pd.DataFrame(tune).T.sort_values("MAE_moy").round(3)
display(tune_tab)

BEST_KEY = tune_tab.index[0].split("_")[1]
BEST_PARAMS = grid[BEST_KEY]
print("Réglage retenu :", BEST_KEY, BEST_PARAMS)

## Étape 8. Modèles finaux et test final (une seule fois)

On entraîne **quatre** modèles LightGBM avec le réglage retenu, car chacun répond à une question différente :

| Modèle | Ce qu'il prédit | Usage |
|---|---|---|
| Médiane | La durée « typique » (la moitié des séjours sont plus courts) | Chiffre central pour un patient |
| 10e percentile | Une durée basse | Bas de la fourchette |
| 90e percentile | Une durée haute | Haut de la fourchette |
| Moyenne | La durée moyenne attendue | **Calcul des lits** : c'est la moyenne, pas la médiane, qui s'additionne |

On évalue **une seule fois** sur les 20 % de patients mis de côté (retours en moins de 30 jours uniquement) :

- **MAE** de la médiane, comparée à « toujours la médiane de l'entraînement » ;
- **couverture de la fourchette** : la fourchette 10 % à 90 % doit contenir environ 80 % des durées réelles. Comme les modèles de quantiles donnent souvent une fourchette trop étroite, on l'élargit d'un nombre de jours fixe (`DELTA`), appris sur `dev` ;
- **erreur sur un groupe de 100 retours** : on compare la somme des jours prévus à la somme réelle. C'est le test le plus important pour la planification des lits.

In [ ]:
tr_mask = np.ones(len(X_dev), bool) if SCOPE == "all" else (lt_dev == 1)
Xtr, ytr = X_dev[tr_mask], y_dev[tr_mask]
te = lt_test == 1
Xte, yte = X_test[te], y_test[te]
print("Entraînement :", len(Xtr), "couples | Test :", len(Xte), "retours < 30 jours")

m_med = make_lgbm(BEST_PARAMS, "l1").fit(Xtr, ytr)
m_q10 = make_lgbm(BEST_PARAMS, "quantile", 0.10).fit(Xtr, ytr)
m_q90 = make_lgbm(BEST_PARAMS, "quantile", 0.90).fit(Xtr, ytr)
m_avg = make_lgbm(BEST_PARAMS, "regression").fit(Xtr, ytr)

# --- Calibration de la fourchette sur dev (prédictions hors entraînement) ---
# Les modèles de quantiles donnent souvent une fourchette trop étroite. On cherche l'élargissement
# DELTA (en jours) pour que la fourchette contienne 80 % des durées réelles sur dev.
oof = {k: np.full(len(X_dev), np.nan) for k in ["med", "lo", "hi"]}
for tr, va in StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(X_dev, bucket(y_dev), groups=g_dev):
    tr_ = tr if SCOPE == "all" else tr[lt_dev[tr] == 1]
    for key, obj, a in [("med", "l1", None), ("lo", "quantile", 0.10), ("hi", "quantile", 0.90)]:
        oof[key][va] = make_lgbm(BEST_PARAMS, obj, a).fit(X_dev.iloc[tr_], y_dev[tr_]).predict(X_dev.iloc[va])
ev = lt_dev == 1
o_med = np.clip(oof["med"][ev], 1, 14)
o_lo = np.minimum(np.clip(oof["lo"][ev], 1, 14), o_med)
o_hi = np.maximum(np.clip(oof["hi"][ev], 1, 14), o_med)
DELTA = 0.0
for d in np.arange(0, 6.01, 0.05):
    if ((y_dev[ev] >= np.clip(o_lo - d, 1, 14)) & (y_dev[ev] <= np.clip(o_hi + d, 1, 14))).mean() >= 0.80:
        DELTA = float(d); break
print(f"Élargissement de la fourchette appris sur dev : {DELTA:.2f} jour de chaque côté")

def predict_all(Xd, widen=True):
    med = np.clip(m_med.predict(Xd), 1, 14)
    lo = np.clip(m_q10.predict(Xd), 1, 14)
    hi = np.clip(m_q90.predict(Xd), 1, 14)
    avg = np.clip(m_avg.predict(Xd), 1, 14)
    lo, hi = np.minimum(lo, med), np.maximum(hi, med)    # garantit bas <= médiane <= haut
    if widen:
        lo, hi = np.clip(lo - DELTA, 1, 14), np.clip(hi + DELTA, 1, 14)
    return med, lo, hi, avg

p_med, p_lo, p_hi, p_avg = predict_all(Xte)
_, raw_lo, raw_hi, _ = predict_all(Xte, widen=False)

ref_med = np.median(y_dev[lt_dev == 1])      # référence : toujours la médiane de l'entraînement
ref_avg = y_dev[lt_dev == 1].mean()

err, err_ref = np.abs(p_med - yte), np.abs(ref_med - yte)
err_j = np.abs(np.round(p_med) - yte)         # durées arrondies au jour
print(f"\nMAE médiane du modèle : {err.mean():.3f} jours  | référence (toujours {ref_med:.0f} j) : {err_ref.mean():.3f} jours")
print(f"Gain : {(err_ref.mean() - err.mean()) / err_ref.mean():.1%}")
print(f"Prédictions (arrondies) à ± 1 jour : {(err_j <= 1).mean():.1%} (référence {(err_ref <= 1).mean():.1%}) | "
      f"à ± 2 jours : {(err_j <= 2).mean():.1%} (référence {(err_ref <= 2).mean():.1%})")

cover_raw = ((yte >= raw_lo) & (yte <= raw_hi)).mean()
cover = ((yte >= p_lo) & (yte <= p_hi)).mean()
print(f"\nFourchette 10 %-90 % brute     : contient {cover_raw:.1%} des durées réelles | largeur moyenne {np.mean(raw_hi - raw_lo):.1f} jours")
print(f"Fourchette élargie (retenue)  : contient {cover:.1%} des durées réelles (cible : environ 80 %) | "
      f"largeur moyenne {np.mean(p_hi - p_lo):.1f} jours")

# Erreur sur des groupes de 100 retours (sommes de jours-lits)
rng = np.random.RandomState(SEED)
n_g = min(100, max(10, len(yte) // 4))
s_pred, s_true, s_ref = [], [], []
for _ in range(2000):
    idx = rng.choice(len(yte), n_g, replace=False)
    s_pred.append(p_avg[idx].sum()); s_true.append(yte[idx].sum()); s_ref.append(ref_avg * n_g)
s_pred, s_true, s_ref = map(np.array, (s_pred, s_true, s_ref))
rel, rel_ref = (s_pred - s_true) / s_true, (s_ref - s_true) / s_true
print(f"\nGroupes de {n_g} retours (jours-lits prévus contre réels, 2 000 tirages) :")
print(f"  Modèle     : biais {rel.mean():+.1%} | erreur typique {np.median(np.abs(rel)):.1%} | "
      f"dans 90 % des cas moins de {np.percentile(np.abs(rel), 90):.1%}")
print(f"  Référence  : biais {rel_ref.mean():+.1%} | erreur typique {np.median(np.abs(rel_ref)):.1%} | "
      f"dans 90 % des cas moins de {np.percentile(np.abs(rel_ref), 90):.1%}  (toujours {ref_avg:.1f} j par retour)")

# Classes court / normal / long
true_c, pred_c = bucket(yte), bucket(np.round(p_med))
maj = np.bincount(bucket(y_dev[lt_dev == 1])).argmax()
names = ["court (1-2 j)", "normal (3-6 j)", "long (> 6 j)"]
print(f"\nClasses court / normal / long : précision {(true_c == pred_c).mean():.1%} "
      f"(référence « toujours {names[maj]} » : {(true_c == maj).mean():.1%})")

**Comment lire le résultat :**
- Si la MAE du modèle est à peine plus petite que celle de la référence, la durée de retour est surtout imprévisible avec ces données. C'est un résultat honnête à garder tel quel.
- La couverture de la fourchette **élargie** doit être proche de 80 %. Si elle en est très loin sur le test, il ne faut pas s'y fier.
- **L'erreur sur les groupes de 100 est le chiffre à retenir pour les lits** : les erreurs individuelles se compensent en partie dans un groupe. Un biais proche de 0 % montre que la somme des jours prévus est fiable.

## Étape 9. Où le modèle se trompe

Les modèles de durée ont un défaut classique : ils **tirent les prédictions vers le milieu**. Les séjours courts sont un peu surestimés, les séjours longs sous-estimés. On le vérifie ici, car ce sont justement les séjours longs qui surchargent les lits.

In [ ]:
df_err = pd.DataFrame({"reel": yte, "pred_mediane": p_med, "pred_moyenne": p_avg, "classe": [names[i] for i in true_c]})
by_class = df_err.groupby("classe").agg(retours=("reel", "size"), duree_reelle_moy=("reel", "mean"),
                                         duree_prevue_moy=("pred_moyenne", "mean"))
by_class["MAE_jours"] = df_err.assign(e=np.abs(df_err.pred_mediane - df_err.reel)).groupby("classe")["e"].mean()
display(by_class.loc[[n for n in names if n in by_class.index]].round(2))

print("Matrice de confusion (lignes = réalité, colonnes = prédiction) :")
display(pd.crosstab(pd.Series([names[i] for i in true_c], name="réalité"),
                    pd.Series([names[i] for i in pred_c], name="prédiction")))

# Durée moyenne prévue contre durée moyenne réelle, par dixième de patients
q = min(10, max(2, len(df_err) // 40))
df_err["groupe"] = pd.qcut(df_err["pred_moyenne"].rank(method="first"), q, labels=range(1, q + 1))
cal = df_err.groupby("groupe").agg(prevue=("pred_moyenne", "mean"), reelle=("reel", "mean"), n=("reel", "size"))
display(cal.round(2))
lim = [min(cal.min()[:2]) - 0.3, max(cal.max()[:2]) + 0.3]
plt.figure(figsize=(4.5, 4.5))
plt.plot(lim, lim, "--", color="grey", label="parfait")
plt.plot(cal["prevue"], cal["reelle"], "o-", label="modèle")
plt.xlabel("Durée moyenne prévue (jours)"); plt.ylabel("Durée moyenne réelle (jours)")
plt.title("Durées prévues contre réelles"); plt.legend(); plt.show()

## Étape 10. Importance des variables

On mélange au hasard une variable à la fois et on mesure de combien la MAE **augmente** : plus l'augmentation est grande, plus le modèle dépend de cette variable.

In [ ]:
from sklearn.inspection import permutation_importance

pi = permutation_importance(m_med, Xte, yte, scoring="neg_mean_absolute_error",
                            n_repeats=5, random_state=SEED, n_jobs=1)
imp = pd.Series(pi.importances_mean, index=Xte.columns).sort_values(ascending=False)
imp.head(12).iloc[::-1].plot.barh(figsize=(6, 4), title="Variables les plus utiles (hausse de la MAE en jours)")
plt.show()
imp.head(12).round(4)

## Étape 11. Sauvegarde, utilisation et charge attendue

La charge de lits d'un patient est : **probabilité de retour (modèle 1) × durée moyenne prévue (modèle 2)**. L'exemple ci-dessous utilise des probabilités **imaginées** (illustratif) : en vrai, elles viennent du modèle 1.

In [ ]:
artifact = dict(m_med=m_med, m_q10=m_q10, m_q90=m_q90, m_avg=m_avg, cat=CAT, num=NUM, categories=CATEGORIES,
                top_spec=TOP_SPEC, scope=SCOPE, best_params=BEST_PARAMS, ref_med=float(ref_med), ref_avg=float(ref_avg), delta=float(DELTA))
joblib.dump(artifact, "model_duree_sejour_retour.joblib")
print("Modèle sauvegardé : model_duree_sejour_retour.joblib")
try:
    from google.colab import files
    files.download("model_duree_sejour_retour.joblib")
except ImportError:
    pass

def predict_los(raw_df):
    """raw_df : séjours (mêmes colonnes que diabetic_data.csv). Renvoie la durée du séjour de retour, SI le patient revient."""
    d = raw_df.copy()
    d["is_lt30"] = 1
    Xn = apply_categories(build_features(d, retour_lt30=1))
    med, lo, hi, avg = predict_all(Xn)
    return pd.DataFrame({"duree_mediane_j": med.round(1), "duree_basse_j": lo.round(1),
                         "duree_haute_j": hi.round(1), "duree_moyenne_j": avg.round(1)}, index=raw_df.index)

def charge_attendue(p_retour, los_moyenne):
    """Jours-lits attendus pour un patient = probabilité de retour x durée moyenne prévue."""
    return np.asarray(p_retour) * np.asarray(los_moyenne)

# Exemple : 5 patients du test, avec des probabilités de retour IMAGINÉES (illustratif)
ex = pairs.iloc[test_idx].loc[lambda d: d["is_lt30"] == 1].head(5)
pred = predict_los(ex)
pred["p_retour_exemple"] = [0.32, 0.14, 0.05, 0.20, 0.10]
pred["charge_attendue_jours_lits"] = charge_attendue(pred["p_retour_exemple"], pred["duree_moyenne_j"]).round(2)
pred